# ChatGPT Twitter Sentiment Analysis (500K Tweets)

This notebook is a recruiter-friendly walkthrough of the production pipeline in `src/`. Place the Kaggle CSV at `data/raw/chatgpt_tweets.csv` before running.

## 1. Imports and Setup

In [ ]:
from pathlib import Path
import pandas as pd

from src.config import RAW_DATA_PATH, CLEANED_DATA_PATH, FEATURED_DATA_PATH, IMAGE_DIR
from src.preprocessing import clean_tweets
from src.features import add_all_features
from src.sentiment import add_sentiment_features
from src.nlp_evaluation import compare_sentiment_models, plot_confusion_matrix
from src.visualization import generate_core_visualizations
from src.utils import safe_read_csv, setup_logging

setup_logging()

## 2. Load Raw Data

In [ ]:
raw_df = safe_read_csv(RAW_DATA_PATH)
raw_df.head()

## 3. Data Cleaning

In [ ]:
cleaned_df = clean_tweets(raw_df)
cleaned_df.to_csv(CLEANED_DATA_PATH, index=False)
cleaned_df[['tweet_date', 'username', 'country', 'is_verified', 'clean_text', 'hashtags', 'mentions']].head()

## 4. Feature Engineering and Sentiment

In [ ]:
featured_df = add_sentiment_features(add_all_features(cleaned_df))
featured_df.to_csv(FEATURED_DATA_PATH, index=False)
featured_df[['tweet_length', 'word_count', 'hashtag_count', 'mention_count', 'compound_score', 'sentiment_label', 'day', 'month', 'hour', 'week', 'weekend_flag']].head()

## 5. NLP Model Comparison

In [ ]:
metrics = compare_sentiment_models(featured_df)
metrics

In [ ]:
plot_confusion_matrix(featured_df, IMAGE_DIR / 'generated' / 'sentiment_confusion_matrix.png')

## 6. Generate EDA Visualizations

In [ ]:
generated_paths = generate_core_visualizations(featured_df, IMAGE_DIR / 'generated')
len(generated_paths), generated_paths[:5]

## 7. Business Takeaways

- Pair sentiment trend changes with tweet spikes to identify likely event-driven public reaction.
- Use minimum tweet thresholds before ranking countries or hashtags by sentiment ratio.
- Compare verified and non-verified users because public figures can drive a different conversation than general users.
- Separate excitement topics such as coding and productivity from concern topics such as privacy, job loss, education, and misinformation.